In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, to_date, when, count

# Initialize Spark
spark = SparkSession.builder.appName("CustomerOrderInsights").getOrCreate()


In [0]:
from pyspark.sql import Row

# Sample orders data
orders_data = [
    Row(customer_id=1, order_date="2026-01-05", delivery_date="2026-01-08"),
    Row(customer_id=2, order_date="2026-01-10", delivery_date="2026-01-20"),
    Row(customer_id=3, order_date="2026-02-01", delivery_date="2026-02-03"),
    Row(customer_id=1, order_date="2026-02-15", delivery_date="2026-02-28"),
    Row(customer_id=4, order_date="2026-03-01", delivery_date="2026-03-04"),
    Row(customer_id=2, order_date="2026-03-10", delivery_date="2026-03-12"),
    Row(customer_id=5, order_date="2026-03-20", delivery_date="2026-04-01"),
    Row(customer_id=3, order_date="2026-04-05", delivery_date="2026-04-07"),
]
orders_df = spark.createDataFrame(orders_data)

# Sample customers data
customers_data = [
    Row(customer_id=1, region="North"),
    Row(customer_id=2, region="South"),
    Row(customer_id=3, region="East"),
    Row(customer_id=4, region="West"),
    Row(customer_id=5, region="North"),
]
customers_df = spark.createDataFrame(customers_data)

In [0]:
# Convert dates
orders_df = orders_df.withColumn("order_date", to_date(col("order_date"))) \
                     .withColumn("delivery_date", to_date(col("delivery_date")))


In [0]:
# Calculate delay
orders_df = orders_df.withColumn(
    "delay_days", datediff(col("delivery_date"), col("order_date"))
).withColumn(
    "delayed", when(col("delay_days") > 5, 1).otherwise(0)
)


In [0]:
# Join with customers
joined_df = orders_df.join(customers_df, on="customer_id", how="left")

In [0]:
# Group by region and count delays
region_delays = joined_df.groupBy("region").agg(
    count(when(col("delayed") == 1, True)).alias("delayed_orders"),
    count("*").alias("total_orders")
)

region_delays.show()

+------+--------------+------------+
|region|delayed_orders|total_orders|
+------+--------------+------------+
| North|             2|           3|
| South|             1|           2|
|  East|             0|           2|
|  West|             0|           1|
+------+--------------+------------+



In [0]:
import pandas as pd

pdf = region_delays.toPandas()
display(pdf)

region,delayed_orders,total_orders
North,2,3
South,1,2
East,0,2
West,0,1
